# Exercise 9 — Sound and music description

In this exercise you describe sounds using simple machine-learning workflows with Freesound descriptors. You will use the Freesound API to download short instrumental sounds and pre-computed descriptors, then apply unsupervised and supervised methods for organisation and classification.

You will complete **four parts**:
1. Download sound collections and descriptor files from Freesound.
2. Select a descriptor pair for 2D visual clustering.
3. Cluster sounds with k-means using multiple descriptors.
4. Classify query sounds with k-NN and interpret the result.

The notebook already provides most of the implementation, so focus on selecting good sounds, choosing meaningful descriptors, and explaining your decisions.

### Relevant concepts

#### Freesound API
The Freesound API allows programmatic search and retrieval of sounds, metadata, and analysis descriptors. It supports text queries, tag filters, and duration constraints, similar to advanced search on the website. Documentation: https://freesound.org/docs/api/

#### Sound descriptors
You will use the current Freesound analysis fields, also used in the Freesound example notebooks. Each descriptor is a recording-level summary computed with Essentia. These descriptors summarise spectral and temporal characteristics (e.g., spectral centroid, dissonance, attack time, inharmonicity, MFCCs), which can be used as features for clustering and classification.

#### Euclidean distance
The Euclidean distance between two descriptor vectors $p$ and $q$ is:

$$
d(p,q) = \sqrt{\sum_{i=1}^{n}(q_i - p_i)^2}
$$

This distance is used as a similarity measure in both clustering and k-NN classification.

#### K-means clustering
K-means partitions data into $k$ clusters by minimising the sum of squared distances to the assigned centroids. Each sound is assigned to the nearest centroid in feature space. For this exercise, use $k=3$ (one cluster per chosen instrument class).

#### K-nearest neighbours (k-NN)
k-NN predicts the class of a query sample by majority vote among its $K$ nearest reference sounds in descriptor space. With $K=1$, classification is the label of the nearest neighbour.

## Part 1 – Download sounds and descriptors from Freesound

Build three collections of **20 valid recordings each**. Choose three instruments from violin, guitar, bassoon, trumpet, clarinet, cello and naobo. Aim for one clearly audible note or stroke per recording, under 10 seconds.

**Setup.** In the active Jupyter kernel, install the packages with `%pip install numpy scipy matplotlib requests python-dotenv` if needed. Restart the kernel after installing. Get a [Freesound API key](https://freesound.org/apiv2/apply/) and set `FREESOUND_API_KEY` in your environment or private `.env` file. If needed, the notebook prompts for it without displaying it.

**E9 – 1.1.** Try your query on the Freesound website first. In the answer cell, call `download_class_sounds(session, query, tag, count, target_root, duration)` for each instrument. The arguments correspond to the report fields `queryText`, `tag` and `duration`. Use `count=20` and an upper duration limit below 10 s. Use `None` for an unfiltered tag.

The download helper from the classification example caches `features.json`, `metadata.json` and, when available, `preview.mp3` under `<target_root>/<instrument>/<sound ID>/`. A cached descriptor passes a numeric check. Listen to the preview or use the link in the metadata to check the sound itself. Keep a record of which of the first 20 results you accept and why you reject others.

Use a **separate target root for each search attempt** so a refined search starts with its own cache. Copy or move only the accepted sound folders into `targetDir/<instrument>/` for analysis. Refine the search to replace rejected or duplicate recordings. Keep all search-attempt caches outside `targetDir`.

In [ ]:
from collections import Counter
from getpass import getpass
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from scipy.cluster.vq import kmeans, vq, whiten
from urllib3.util.retry import Retry

# Use the current API names and feature selection from the classification example.
SCALAR_DESCRIPTORS = [
    'spectral_centroid', 'spectral_energy', 'dissonance',
    'pitch_salience', 'inharmonicity', 'log_attack_time',
]
feature_names = SCALAR_DESCRIPTORS + [f'mfcc.{i}' for i in range(13)]
descriptorMapping = dict(enumerate(feature_names))
API_DESCRIPTORS = SCALAR_DESCRIPTORS + ['mfcc']

targetDir = Path('testDownload') / 'collections'
queryDir = Path('testDownload') / 'queries'


def freesound_session():
    load_dotenv()
    api_key = (os.getenv('FREESOUND_API_KEY') or getpass('Freesound API key: ')).strip()
    if not api_key:
        raise ValueError('Enter a Freesound API key to download sounds.')
    return make_session(api_key)

In [ ]:
BASE_URL = 'https://freesound.org/apiv2'


def make_session(api_key):
    session = requests.Session()
    session.headers.update({'Authorization': f'Token {api_key}'})
    # Retry occasional server/rate-limit failures, respecting Retry-After.
    retries = Retry(total=3, backoff_factor=0.5,
                    status_forcelist=[429, 500, 502, 503, 504],
                    allowed_methods=['GET'], respect_retry_after_header=True)
    session.mount('https://', HTTPAdapter(max_retries=retries))
    return session


def api_get_json(session, endpoint, *, params=None):
    url = endpoint if endpoint.startswith('https://freesound.org/apiv2/') else BASE_URL + endpoint
    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


def feature_vector(analysis, names):
    """Convert current APIv2 descriptor JSON to a selected 1D numerical vector."""
    values = []
    for name in names:
        if '.' in name:  # For example, mfcc.3
            descriptor, index = name.rsplit('.', 1)
            vector = analysis.get(descriptor)
            if not isinstance(vector, (list, tuple)) or len(vector) <= int(index):
                raise ValueError(f'Missing/short descriptor: {descriptor}')
            value = vector[int(index)]
        else:
            value = analysis.get(name)
        if value is None or isinstance(value, (list, tuple, dict, bool)):
            raise ValueError(f'Missing or non-scalar descriptor: {name}')
        value = float(value)
        if not np.isfinite(value):
            raise ValueError(f'Non-finite descriptor: {name}')
        values.append(value)
    return np.asarray(values, dtype=float)


def download_preview(preview_url, target):
    """Previews may be hosted externally: do NOT send the API token to that URL."""
    if not preview_url:
        return
    with requests.get(preview_url, stream=True, timeout=30) as response:
        response.raise_for_status()
        with target.open('wb') as handle:
            for chunk in response.iter_content(chunk_size=64 * 1024):
                if chunk:
                    handle.write(chunk)


def download_class_sounds(session, query, tag, count, target_root, duration,
                          save_previews=True, max_candidates=200, cache_folder=None):
    """Cache at most `count` valid sounds per class, without deleting existing files."""
    class_dir = target_root / (cache_folder or query)
    class_dir.mkdir(parents=True, exist_ok=True)
    existing_ids = set()
    for path in class_dir.glob('*/features.json'):
        try:
            feature_vector(json.loads(path.read_text(encoding='utf-8')), feature_names)
        except (ValueError, TypeError):
            continue
        existing_ids.add(path.parent.name)
    if len(existing_ids) >= count:
        print(f'{query}: {len(existing_ids)} cached sounds; no download needed.')
        return

    filters = [f'duration:[{duration[0]} TO {duration[1]}]']
    if tag:
        filters.insert(0, f'tag:{tag}')

    page = 1
    seen = 0
    added = 0
    skipped = 0
    while len(existing_ids) < count and seen < max_candidates:
        payload = api_get_json(session, '/search/', params={
            'query': query,
            'filter': ' '.join(filters),
            'sort': 'score',
            'fields': 'id,name,url,username,license,previews',
            'page_size': 30,
            'page': page,
        })
        results = payload.get('results', [])
        if not results:
            break
        for sound in results:
            if len(existing_ids) >= count or seen >= max_candidates:
                break
            seen += 1
            sound_id = str(sound['id'])
            if sound_id in existing_ids:
                continue
            sound_dir = class_dir / sound_id
            feature_file = sound_dir / 'features.json'
            try:
                analysis = api_get_json(session, f'/sounds/{sound_id}/analysis/',
                                        params={'fields': ','.join(API_DESCRIPTORS)})
                # Require the lab's descriptor fields so all experiments use the same sounds.
                feature_vector(analysis, feature_names)
            except requests.HTTPError as exc:
                if exc.response.status_code != 404:
                    raise
                skipped += 1
                if skipped <= 3:
                    print(f'  No analysis for {sound_id}.')
                continue
            except (ValueError, KeyError, TypeError) as exc:
                skipped += 1
                if skipped <= 3:
                    print(f'  Skipping {sound_id}: {type(exc).__name__}: {exc}')
                continue

            sound_dir.mkdir(parents=True, exist_ok=True)
            with feature_file.open('w', encoding='utf-8') as f:
                json.dump(analysis, f, indent=2)
            with (sound_dir / 'metadata.json').open('w', encoding='utf-8') as f:
                json.dump({k: sound.get(k) for k in
                           ['id', 'name', 'url', 'username', 'license']}, f, indent=2)
            if save_previews:
                preview = (sound.get('previews') or {}).get('preview-lq-mp3')
                try:
                    download_preview(preview, sound_dir / 'preview.mp3')
                except requests.RequestException as exc:
                    (sound_dir / 'preview.mp3').unlink(missing_ok=True)
                    print(f'  Preview unavailable for {sound_id}: {type(exc).__name__}')
            existing_ids.add(sound_id)
            added += 1
        if not payload.get('next'):
            break
        page += 1
    print(f'{query}: {len(existing_ids) - added} cached + {added} new; {skipped} incomplete/failed.')


def load_dataset(root, selected_names, class_filter=None):
    features, labels, ids, paths = [], [], [], []
    seen_ids = set()
    # Only files named features.json from this API version.
    for file_path in sorted(Path(root).glob('*/*/features.json')):
        if class_filter and file_path.parent.parent.name != class_filter:
            continue
        if file_path.parent.name in seen_ids:
            print(f'Skipping duplicate Freesound ID {file_path.parent.name} to prevent data leakage.')
            continue
        try:
            analysis = json.loads(file_path.read_text(encoding='utf-8'))
            vector = feature_vector(analysis, selected_names)
        except (ValueError, TypeError, KeyError, json.JSONDecodeError) as exc:
            print(f'Skipping {file_path}: {exc}')
            continue
        features.append(vector)
        labels.append(file_path.parent.parent.name)
        ids.append(file_path.parent.name)
        paths.append(file_path)
        seen_ids.add(file_path.parent.name)
    if not features:
        return np.empty((0, len(selected_names))), np.array([]), [], []
    return np.vstack(features), np.asarray(labels), ids, paths

In [ ]:
# E9 – 1.1: configure and download the three instrument searches.
# Example call (uncomment and adapt):
# with freesound_session() as session:
#     download_class_sounds(session, 'violin', 'pizzicato', 20,
#                           Path('testDownload') / 'attempt-1', (0, 3))
# Add your own calls and curate targetDir before continuing.
### your code here

**E9 – 1.2: Conceptual questions on collection coherence (answer here)**

1. For each of your three instruments, report the final query settings (`queryText`, `tag`, `duration`) and explain why these settings produce coherent single-note/single-stroke samples.
2. Of the first 20 downloaded sounds for each instrument, how many did you accept after listening? If you rejected any samples, describe the most common reasons (e.g., polyphonic texture, heavy effects, wrong instrument, too long, poor recording quality).
3. Compare at least two query attempts for one instrument (initial vs refined). What changed in the results after modifying `tag` or `duration`, and why did those changes improve coherence?
4. After making your plots in Part 2, did you observe descriptor outliers within an instrument set? Pick one outlier by Freesound ID and explain why its descriptor profile might differ from the rest of its class.
5. Why is collection coherence critical before running the Part 2 visual comparison and Part 3 k-means? Describe one concrete failure mode in clustering/classification that would appear if your downloaded collections include many mislabeled or heterogeneous samples.
6. If you had to improve coherence further without reducing diversity too much, what additional metadata filters would you add (e.g., license, samplerate, channels, tags), and why?

## Part 2 – Select two descriptors for 2D clustering

Use the curated `targetDir` from Part 1. It should contain only the three instrument folders. In each plot, a dot is one sound and its colour is the collection label. This step is a visual comparison before fitting k-means.

Try at least four pairs of indices from `descriptorMapping`. Prefer a pair with tight groups within each label and little overlap between labels. Set `annotate=True` to show Freesound IDs and `normalize=True` to compare axes in standard-deviation units. Indices refer to the current API fields and differ from the legacy mapping.

In [ ]:
# The lab uses the six scalar descriptors and all 13 MFCCs from the current API.
# MFCC 0 strongly reflects level; the other coefficients describe spectral shape.
for index, name in descriptorMapping.items():
    print(f'{index:2}  {name}')

In [ ]:
def select_descriptors(indices):
    if not len(indices) or len(set(indices)) != len(indices):
        raise ValueError('Choose a non-empty set of distinct descriptor indices.')
    try:
        return [descriptorMapping[index] for index in indices]
    except KeyError as exc:
        raise ValueError(f'Unknown descriptor index: {exc.args[0]}') from exc


def reference_dataset(targetDir, descInput):
    names = select_descriptors(descInput)
    matrix, labels, ids, paths = load_dataset(targetDir, names)
    if len(set(labels)) != 3:
        raise ValueError('Use three non-empty instrument folders inside targetDir.')
    return matrix, labels, ids, names


def plot_descriptor_pair(targetDir, descInput, *, annotate=False, normalize=False):
    if len(descInput) != 2:
        raise ValueError('Select exactly two descriptors for a 2D plot.')
    matrix, labels, ids, names = reference_dataset(targetDir, descInput)
    if normalize:
        matrix = whiten(matrix)
    fig, ax = plt.subplots(figsize=(9, 5))
    for label in sorted(set(labels)):
        selected = labels == label
        ax.scatter(*matrix[selected].T, label=label, alpha=0.75)
    if annotate:
        for (x, y), sound_id in zip(matrix, ids):
            ax.annotate(sound_id, (x, y), fontsize=8)
    suffix = ' / σ' if normalize else ''
    ax.set(xlabel=names[0] + suffix, ylabel=names[1] + suffix)
    ax.legend()
    plt.show()

In [ ]:
# E9 – 2.1: choose a pair and repeat with at least three more pairs.
descInput = (0, 4)
plot_descriptor_pair(targetDir, descInput, annotate=False)

**E9 – 2.2: Conceptual questions on descriptor choice (answer here)**

1. List at least four descriptor pairs you tested and briefly describe the visual clustering quality of each pair (compactness within class and separation between classes).
2. For the pair you selected, explain what each descriptor measures perceptually (e.g., brightness, attack, noisiness, harmonicity) and why that makes it suitable for separating your three instrument classes.
3. Identify one instrument pair that still overlaps in your selected 2D plot. What acoustic or timbral similarity could explain the overlap? If there is none, examine the closest pair of classes. Suggest one additional descriptor that could help separate them and justify your choice.
4. Compare the raw and normalised plots. Did rescaling the axes or inspecting outliers change your visual interpretation? Pick one outlier point, give its sound ID, and explain whether it is a true timbral exception, a mislabeled sample, or a recording artifact.
5. If you were forced to choose a different descriptor pair (second-best), which one would you pick and why? What trade-off would you accept compared with your best pair?

## Part 3 – Cluster sounds with k-means

Use the same curated dataset. Set `nCluster=3` and compare at least three subsets with more than two descriptors. Start with your selected pair and add features. Compare subsets with the same seed, then repeat the best configuration with at least five different seeds.

As in the clustering example, SciPy `whiten` divides each feature by its standard deviation without subtracting the mean. K-means fits the scaled vectors without using the instrument labels. Afterwards, the code assigns each cluster its majority label and reports the fraction of sounds that agree with those labels. This is an in-sample clustering score (purity). Multiple clusters can share a majority label.

In [ ]:
def cluster_sounds(targetDir, nCluster, descInput, *, seed=None):
    matrix, labels, ids, names = reference_dataset(targetDir, descInput)
    if not 2 < len(names):
        raise ValueError('Choose more than two descriptors for Part 3.')
    if not 1 <= nCluster <= len(matrix):
        raise ValueError('nCluster must be between 1 and the number of sounds.')

    scaled = whiten(matrix)
    centers, _ = kmeans(scaled, nCluster, seed=seed)
    assignments, _ = vq(scaled, centers)
    rows = []
    if len(centers) < nCluster:
        print(f'Only {len(centers)} non-empty clusters were found.')
    for index in range(len(centers)):
        members = np.flatnonzero(assignments == index)
        if not len(members):
            print(f'Cluster {index}: no assigned sounds')
            continue
        counts = Counter(labels[members])
        majority = min(counts, key=lambda label: (-counts[label], label))
        print(f'Cluster {index}: {dict(sorted(counts.items()))} → {majority}')
        rows.extend({
            'sound_id': ids[j], 'label': labels[j], 'cluster': index,
            'majority_label': majority, 'agrees': labels[j] == majority,
        } for j in members)
    purity = float(np.mean([row['agrees'] for row in rows]))
    print(f'Agreement with cluster majority labels: {purity:.1%}')
    return {'purity': purity, 'sounds': rows}

In [ ]:
# E9 – 3.1: test at least three subsets. Add a descriptor to your Part 2 pair.
# results = cluster_sounds(targetDir, 3, [0, 4, 5], seed=0)
# Inspect results['sounds'] for individual recordings.
### your code here

**E9 – 3.2: Conceptual questions (answer here)**

1. List the descriptor subsets you tested for k-means (at least three combinations), and report the percentage agreement with cluster majority labels (purity) for each run. Which subset gave the best result, and what is the minimum number of descriptors that still kept accuracy close to your best run?
2. Inspect the confusion pattern in your best clustering result. Which instrument pair was most frequently mixed up, and which acoustic/timbral properties of those instruments explain that overlap in descriptor space?
3. K-means uses random centroid initialisation. Run your best descriptor configuration at least 5 times and report the mean and range of purity. Did performance vary significantly? Based on your results, is your descriptor set robust or sensitive to initialisation?
4. Pick one sound whose label disagrees with its cluster majority and listen to it. Does it acoustically resemble the class where it was assigned more than its original class? Use at least two descriptors to justify why its position in feature space may be ambiguous.

## Part 4 – Classify sounds with k-NN

Download queries into `queryDir`, separate from your reference collections. Use at least one sound from outside the three instrument classes. Use `classify_sound_kNN()` to classify each query by Euclidean distance on selected raw descriptors. Large numerical ranges can dominate that distance.

Run at least five experiments, varying the query, descriptor subset and `K`. Keep the query and descriptors fixed for at least one small-versus-large `K` comparison so you can isolate its effect. Include both an expected prediction and a surprising one.

In [ ]:
def compute_similar_sounds(queryFile, targetDir, descInput):
    matrix, labels, ids, names = reference_dataset(targetDir, descInput)
    query_path = Path(queryFile)
    if query_path.parent.name in ids:
        raise ValueError('Use a query sound ID absent from the reference collections.')
    query = feature_vector(json.loads(query_path.read_text(encoding='utf-8')), names)
    distances = np.linalg.norm(matrix - query, axis=1)
    order = np.argsort(distances, kind='stable')
    return [(float(distances[i]), ids[i], labels[i]) for i in order]


def classify_sound_kNN(queryFile, targetDir, K, descInput):
    ranked = compute_similar_sounds(queryFile, targetDir, descInput)
    if not isinstance(K, (int, np.integer)) or not 1 <= K <= len(ranked):
        raise ValueError(f'Choose an integer K between 1 and {len(ranked)}.')
    votes = [label for _, _, label in ranked[:K]]
    counts = Counter(votes)
    # Resolve tied vote counts using the closest neighbour among tied labels.
    prediction = max(votes, key=counts.get)
    print(f'Prediction: {prediction}. Votes: {dict(counts)}')
    return prediction


def nearest_class_gap(ranked):
    # Use the nearest single recording within each label as the class distance.
    nearest = {}
    for distance, sound_id, label in ranked:
        nearest.setdefault(label, (distance, sound_id))
    first, second = list(nearest.items())[:2]
    return first, second, second[1][0] - first[1][0]

In [ ]:
# E9 – 4.1: download a query separately. Adapt this example to your dataset.
# with freesound_session() as session:
#     download_class_sounds(session, 'viola', 'pizzicato', 1, queryDir, (0, 3))
### your code here

**E9 – 4.2: Classification setup (answer here)**

1. For each of your 5 k-NN experiments, report: query sound ID/link, descriptor subset, value of `K`, and predicted class. Keep this in a compact table.
2. Why did you choose those descriptors for k-NN? For each descriptor, explain what perceptual property it captures (e.g., brightness, noisiness, attack sharpness) and why that property should help discriminate your instrument classes.
3. How did changing `K` affect prediction stability when the query and descriptors stayed fixed? Compare at least one small `K` (e.g., 1 or 3) with one larger `K` (e.g., 7 or 9). Which gave more consistent results, and why?
4. Include at least one query sound that is outside your three training classes. Why is it mapped to the predicted class? Identify which training-class timbral characteristics the query most closely matches.

In [ ]:
# E9 – 4.3: replace SOUND_ID with the actual cached ID.
# queryFile = queryDir / 'viola' / 'SOUND_ID' / 'features.json'
# chosen = [0, 4, 5]
# prediction = classify_sound_kNN(queryFile, targetDir, K=3, descInput=chosen)
# ranked = compute_similar_sounds(queryFile, targetDir, chosen)
# print(ranked[:5])
# print(nearest_class_gap(ranked))
### your code here

**E9 – 4.4: Classification interpretation (answer here)**

1. For one correct prediction and one incorrect/surprising prediction, inspect the nearest neighbours returned by the distance metric. Are the top neighbours perceptually plausible? Explain using both listening and descriptor values.
2. For each case, find the minimum neighbour distance within each class. Report the gap between the two smallest class distances using `nearest_class_gap(ranked)`. This gap describes proximity, while the prediction for larger `K` depends on votes. Was the incorrect prediction a borderline decision (small gap) or a clear mismatch (large gap)?
3. If you could add one more descriptor to improve the weakest experiment, which one would you add (from the mapping) and why? Predict how it would change the decision boundary between the two most confusable classes.
4. Reflect on the limits of descriptor-based k-NN in this setup: what kinds of musical variations (articulation, recording conditions, effects, pitch register) are likely to break classification even if the instrument identity is obvious to a listener?